# Cost model and budget simulation

The simulation does not call a model. It spends a counter of 100000. A skip costs 0. A solvable attempt adds `reward` and spends `cost_hat`. An unsolvable attempt spends `cost_hat` and adds 0.

`test-100` is one pass. The train mix is scored as 30 random packs of 100 tasks, because 100000 only covers the head of 3000 tasks.


In [6]:
%pip install -q scikit-learn==1.9.1


Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
import importlib.util
import json

SKLEARN_PIN = "scikit-learn==1.9.1"

# Kaggle mounts this dataset as
# /kaggle/input/datasets/<user>/<slug>/<pack>/<pack>/<task>/task.json
# Local runs still use task_packs/.
SLUGS = {
    "train": ("train-3000", "train_3000"),
    "abstention": ("abstention-pack", "abstention_pack"),
    "test": ("test-100", "test_100"),
}


def _repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "harness" / "features.py").is_file() and (cand / "task_packs").is_dir():
            return cand
    return None


def _task_jsons(root: Path) -> list[Path]:
    direct = sorted(p for p in root.glob("*/task.json") if p.is_file())
    if direct:
        return direct
    return sorted(p for p in root.glob("*/*/task.json") if p.is_file())


def _matches_slug(path: Path, slugs: tuple[str, ...]) -> bool:
    blob = path.as_posix().lower()
    return any(slug in blob for slug in slugs)


def find_pack(key: str) -> Path:
    slugs = SLUGS[key]
    roots = []
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        roots.append(kaggle)
    repo = _repo_root()
    if repo is not None:
        roots.append(repo / "task_packs")
    best = None
    best_n = -1
    for root in roots:
        if not root.is_dir():
            continue
        for manifest in root.rglob("manifest.jsonl"):
            parent = manifest.parent
            if not _matches_slug(parent, slugs):
                continue
            n = len(_task_jsons(parent))
            if n > best_n:
                best = parent
                best_n = n
    if best is None or best_n <= 0:
        raise FileNotFoundError(
            f"{key} pack not found under /kaggle/input or task_packs. "
            f"Expected a manifest.jsonl whose path contains one of {slugs}."
        )
    return best


_EMBEDDED_FEATURES = "\"\"\"Tabular features and structural skip flags. No LLM, no expected/reward.\n\nStandalone: notebooks load this file by path, so it must not import the harness\npackage (that pulls openJiuwen).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport ast\nimport re\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nFEATURE_SCHEMA_VERSION = 1\n\n# Bands match harness.estimate. The heuristic stays here so Kaggle and the\n# runtime share one formula when no cost model is loaded.\nCOST_SHORT = 800\nCOST_CODING = 2500\n\nFEATURE_NAMES: tuple[str, ...] = (\n    \"is_math\",\n    \"is_data_analysis\",\n    \"is_search\",\n    \"is_coding\",\n    \"instr_chars\",\n    \"instr_words\",\n    \"instr_lines\",\n    \"digit_ratio\",\n    \"n_numbers\",\n    \"n_questions\",\n    \"has_latex\",\n    \"has_code_fence\",\n    \"mentions_answer_txt\",\n    \"da_named_file_missing\",\n    \"da_no_csv\",\n    \"da_broken_header\",\n    \"da_column_missing\",\n    \"search_kb_empty\",\n    \"search_kb_tiny\",\n    \"search_token_overlap\",\n    \"coding_no_mylib\",\n    \"coding_empty_test\",\n    \"coding_foreign_import\",\n    \"coding_ast_fail\",\n    \"math_arith_steps\",\n    \"csv_lines\",\n    \"csv_cols\",\n    \"kb_files\",\n    \"kb_chars\",\n    \"code_lines\",\n    \"code_defs\",\n    \"code_asserts\",\n)\n\n# Hard skip only when a missing or wrong file makes the task impossible.\n# A syntax error in mylib.py is a feature, not a skip: it may be the bug to fix.\n# Tiny kb is a feature, not a skip: a short real note would be a false skip.\n_HARD_SKIP_FLAGS: dict[str, tuple[str, ...]] = {\n    \"data_analysis\": (\n        \"da_named_file_missing\",\n        \"da_no_csv\",\n    ),\n    \"search\": (\"search_kb_empty\",),\n    \"coding\": (\n        \"coding_no_mylib\",\n        \"coding_empty_test\",\n        \"coding_foreign_import\",\n    ),\n}\n\n_FILE_RE = re.compile(r\"\\b([\\w./-]+\\.(?:csv|tsv|json|txt|md|py))\\b\", re.IGNORECASE)\n_COL_RE = re.compile(r\"\\bcolumn\\s+([A-Za-z_][\\w]*)\", re.IGNORECASE)\n_NUM_RE = re.compile(r\"\\d+(?:\\.\\d+)?\")\n_WORD_RE = re.compile(r\"[A-Za-z]{3,}\")\n_IMPORT_RE = re.compile(\n    r\"^\\s*(?:from|import)\\s+([A-Za-z_][\\w]*)\",\n    re.MULTILINE,\n)\n_ARITH_RE = re.compile(\n    r\"\\b(?:split|double|doubled|add|added|receive|receives|ship|ships|\"\n    r\"merge|merged|minus|plus|times|divided|subtract|multiply)\\b\",\n    re.IGNORECASE,\n)\n_DEF_RE = re.compile(r\"^\\s*def\\s+\\w+\", re.MULTILINE)\n_ASSERT_RE = re.compile(r\"\\bassert\\b|\\bdef\\s+test_\\w+\", re.IGNORECASE)\n\n_STOP = {\n    \"the\", \"and\", \"for\", \"with\", \"that\", \"this\", \"from\", \"are\", \"was\", \"were\",\n    \"you\", \"your\", \"only\", \"file\", \"answer\", \"when\", \"done\", \"write\", \"final\",\n    \"current\", \"directory\", \"example\", \"bash\", \"echo\", \"python\", \"compute\",\n    \"work\", \"inside\", \"using\", \"use\", \"what\", \"which\", \"who\", \"how\", \"many\",\n    \"does\", \"have\", \"has\", \"into\", \"then\", \"than\", \"after\", \"before\", \"about\",\n}\n\n_CSV_LINE_CAP = 5000\n_KB_CHAR_CAP = 400_000\n_KB_TINY_CHARS = 200\n\n\ndef _task_type(task: Any) -> str:\n    raw = getattr(task, \"task_type\", \"\")\n    return raw.value if hasattr(raw, \"value\") else str(raw)\n\n\ndef _work_dir(task: Any) -> Path:\n    work = getattr(task, \"work_dir\", None)\n    if work is not None:\n        return Path(work)\n    return Path(getattr(task, \"task_dir\"))\n\n\ndef _read_text(path: Path, cap: int) -> str:\n    try:\n        data = path.read_text(encoding=\"utf-8\", errors=\"replace\")\n    except OSError:\n        return \"\"\n    return data[:cap]\n\n\ndef _named_files(instruction: str) -> list[str]:\n    found: list[str] = []\n    for match in _FILE_RE.findall(instruction):\n        if Path(match).name.lower() == \"answer.txt\":\n            continue\n        found.append(match)\n    return found\n\n\ndef _file_exists(work: Path, name: str) -> bool:\n    if (work / name).is_file():\n        return True\n    return (work / Path(name).name).is_file()\n\n\ndef _table_paths(work: Path) -> list[Path]:\n    paths: list[Path] = []\n    for pattern in (\"*.csv\", \"*.tsv\", \"*.json\"):\n        paths.extend(p for p in work.glob(pattern) if p.is_file())\n    return sorted(paths)\n\n\ndef _csv_shape(path: Path) -> tuple[int, int, bool, list[str]]:\n    \"\"\"Return line count (capped), column count, broken-header flag, header cells.\"\"\"\n    try:\n        fh = path.open(\"r\", encoding=\"utf-8\", errors=\"replace\")\n    except OSError:\n        return 0, 0, True, []\n    lines = 0\n    header: list[str] = []\n    with fh:\n        for line in fh:\n            if lines == 0:\n                header = [cell.strip() for cell in line.rstrip(\"\\n\").split(\",\")]\n            lines += 1\n            if lines >= _CSV_LINE_CAP:\n                break\n    broken = lines == 0 or not any(header)\n    cols = 0 if broken else len(header)\n    return lines, cols, broken, header\n\n\ndef _kb_stats(work: Path) -> tuple[int, int, str]:\n    kb = work / \"kb\"\n    if not kb.is_dir():\n        return 0, 0, \"\"\n    files = sorted(p for p in kb.glob(\"*.md\") if p.is_file())\n    chunks: list[str] = []\n    total = 0\n    for path in files:\n        text = _read_text(path, _KB_CHAR_CAP - total)\n        if not text:\n            continue\n        chunks.append(text)\n        total += len(text)\n        if total >= _KB_CHAR_CAP:\n            break\n    body = \"\\n\".join(chunks)\n    nonempty = sum(1 for chunk in chunks if chunk.strip())\n    return nonempty, len(body), body\n\n\ndef _token_overlap(instruction: str, kb_text: str) -> float:\n    question = {w.lower() for w in _WORD_RE.findall(instruction) if w.lower() not in _STOP}\n    if not question or not kb_text:\n        return 0.0\n    kb_words = {w.lower() for w in _WORD_RE.findall(kb_text)}\n    return len(question & kb_words) / len(question)\n\n\ndef _coding_stats(work: Path) -> dict[str, float]:\n    lib = work / \"mylib.py\"\n    test = work / \"test_mylib.py\"\n    lib_text = _read_text(lib, 200_000) if lib.is_file() else \"\"\n    test_text = _read_text(test, 200_000) if test.is_file() else \"\"\n    ast_fail = 0.0\n    if lib.is_file():\n        try:\n            ast.parse(lib_text)\n        except SyntaxError:\n            ast_fail = 1.0\n    imports = {name for name in _IMPORT_RE.findall(test_text)}\n    foreign = 1.0 if imports and \"mylib\" not in imports else 0.0\n    return {\n        \"coding_no_mylib\": 0.0 if lib.is_file() else 1.0,\n        \"coding_empty_test\": 0.0 if test_text.strip() else 1.0,\n        \"coding_foreign_import\": foreign,\n        \"coding_ast_fail\": ast_fail,\n        \"code_lines\": float(lib_text.count(\"\\n\") + (1 if lib_text.strip() else 0)),\n        \"code_defs\": float(len(_DEF_RE.findall(lib_text))),\n        \"code_asserts\": float(len(_ASSERT_RE.findall(test_text))),\n    }\n\n\ndef extract(task: Any) -> dict[str, float]:\n    \"\"\"Feature dict keyed by FEATURE_NAMES. Reward is intentionally absent.\"\"\"\n    task_type = _task_type(task)\n    instruction = str(getattr(task, \"instruction\", \"\") or \"\")\n    work = _work_dir(task)\n    chars = len(instruction)\n    words = re.findall(r\"\\S+\", instruction)\n\n    feats = {name: 0.0 for name in FEATURE_NAMES}\n    feats[\"is_math\"] = 1.0 if task_type == \"math\" else 0.0\n    feats[\"is_data_analysis\"] = 1.0 if task_type == \"data_analysis\" else 0.0\n    feats[\"is_search\"] = 1.0 if task_type == \"search\" else 0.0\n    feats[\"is_coding\"] = 1.0 if task_type == \"coding\" else 0.0\n    feats[\"instr_chars\"] = float(chars)\n    feats[\"instr_words\"] = float(len(words))\n    feats[\"instr_lines\"] = float(instruction.count(\"\\n\") + (1 if instruction else 0))\n    feats[\"digit_ratio\"] = (sum(ch.isdigit() for ch in instruction) / chars) if chars else 0.0\n    feats[\"n_numbers\"] = float(len(_NUM_RE.findall(instruction)))\n    feats[\"n_questions\"] = float(instruction.count(\"?\"))\n    feats[\"has_latex\"] = 1.0 if (\"$\" in instruction or \"\\\\frac\" in instruction) else 0.0\n    feats[\"has_code_fence\"] = 1.0 if \"```\" in instruction else 0.0\n    feats[\"mentions_answer_txt\"] = 1.0 if \"answer.txt\" in instruction.lower() else 0.0\n    feats[\"math_arith_steps\"] = float(len(_ARITH_RE.findall(instruction)))\n\n    if task_type == \"data_analysis\":\n        named = _named_files(instruction)\n        if named and any(not _file_exists(work, name) for name in named):\n            feats[\"da_named_file_missing\"] = 1.0\n        tables = _table_paths(work)\n        if not tables:\n            feats[\"da_no_csv\"] = 1.0\n        else:\n            preferred = tables[0]\n            for name in named:\n                candidate = work / Path(name).name\n                if candidate.is_file():\n                    preferred = candidate\n                    break\n            if preferred.suffix.lower() in {\".csv\", \".tsv\"}:\n                lines, cols, broken, header = _csv_shape(preferred)\n                feats[\"csv_lines\"] = float(lines)\n                feats[\"csv_cols\"] = float(cols)\n                feats[\"da_broken_header\"] = 1.0 if broken else 0.0\n                header_key = {cell.casefold() for cell in header if cell}\n                missing_col = [\n                    col for col in _COL_RE.findall(instruction)\n                    if col.casefold() not in header_key\n                ]\n                if missing_col:\n                    feats[\"da_column_missing\"] = 1.0\n\n    if task_type == \"search\":\n        n_files, n_chars, body = _kb_stats(work)\n        feats[\"kb_files\"] = float(n_files)\n        feats[\"kb_chars\"] = float(n_chars)\n        feats[\"search_kb_empty\"] = 1.0 if n_files == 0 or not body.strip() else 0.0\n        feats[\"search_kb_tiny\"] = 1.0 if 0 < n_chars < _KB_TINY_CHARS else 0.0\n        feats[\"search_token_overlap\"] = _token_overlap(instruction, body)\n\n    if task_type == \"coding\":\n        feats.update(_coding_stats(work))\n\n    return feats\n\n\ndef vector(task: Any) -> list[float]:\n    feats = extract(task)\n    return [float(feats[name]) for name in FEATURE_NAMES]\n\n\ndef structural_skip_reason(task: Any, feats: Optional[dict[str, float]] = None) -> Optional[str]:\n    \"\"\"Comma-joined hard-fail flags, or None. Math never hard-fails.\"\"\"\n    task_type = _task_type(task)\n    flags = _HARD_SKIP_FLAGS.get(task_type)\n    if not flags:\n        return None\n    row = feats if feats is not None else extract(task)\n    hit = [name for name in flags if row.get(name, 0.0) >= 1.0]\n    if not hit:\n        return None\n    return \",\".join(hit)\n\n\ndef heuristic_cost(task: Any, feats: Optional[dict[str, float]] = None) -> int:\n    \"\"\"Same bands as harness.estimate.feature_adjust, from the feature row.\"\"\"\n    row = feats if feats is not None else extract(task)\n    base = COST_CODING if row[\"is_coding\"] >= 1.0 else COST_SHORT\n    factor = 1.0\n    if row[\"instr_chars\"] > 800:\n        factor *= 1.2\n    if row[\"is_search\"] >= 1.0 and row[\"kb_files\"] > 100:\n        factor *= 1.1\n    if row[\"is_data_analysis\"] >= 1.0 and row[\"csv_lines\"] > 500:\n        factor *= 1.1\n    if row[\"is_coding\"] >= 1.0 and row[\"coding_no_mylib\"] < 1.0:\n        factor *= 1.0 + min(row[\"code_lines\"], 200.0) / 200.0\n    return max(1, int(base * factor))\n"

def find_features() -> Path:
    candidates = []
    repo = _repo_root()
    if repo is not None:
        candidates.append(repo / "harness" / "features.py")
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        candidates.extend(kaggle.rglob("features.py"))
    for path in candidates:
        if not path.is_file():
            continue
        head = path.read_text(encoding="utf-8", errors="replace")[:800]
        if "FEATURE_SCHEMA_VERSION" in head:
            return path
    dest = Path("/kaggle/working/features.py")
    if not dest.parent.is_dir():
        dest = Path("features.py")
    dest.write_text(_EMBEDDED_FEATURES, encoding="utf-8")
    print("features.py is not in the Kaggle dataset; using the copy stored in this notebook")
    return dest


PACKS = {key: find_pack(key) for key in SLUGS}
FEATURES_PY = find_features()

spec = importlib.util.spec_from_file_location("rank_features", FEATURES_PY)
features = importlib.util.module_from_spec(spec)
spec.loader.exec_module(features)
print("features", FEATURES_PY, "schema", features.FEATURE_SCHEMA_VERSION)
print({k: str(v) for k, v in PACKS.items()})


def out_dir() -> Path:
    kaggle_out = Path("/kaggle/working")
    if kaggle_out.is_dir():
        return kaggle_out
    return Path.cwd()


OUT = out_dir()
print("artifact dir", OUT)


features.py is not in the Kaggle dataset; using the copy stored in this notebook
features /kaggle/working/features.py schema 1
{'train': '/kaggle/input/datasets/danbrag/agent-dataset/train-3000/train-3000', 'abstention': '/kaggle/input/datasets/danbrag/agent-dataset/abstention-pack/abstention-pack', 'test': '/kaggle/input/datasets/danbrag/agent-dataset/test-100/test-100'}
artifact dir /kaggle/working


In [8]:
def load_pack(root: Path) -> list[dict]:
    manifest = {}
    with (root / "manifest.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            manifest[row["task_id"]] = row
    rows = []
    for task_json in _task_jsons(root):
        raw = json.loads(task_json.read_text(encoding="utf-8"))
        task_dir = task_json.parent
        work = task_dir / "workspace"

        class _Task:
            pass

        task = _Task()
        task.task_id = str(raw["task_id"])
        task.task_type = str(raw["task_type"])
        task.instruction = str(raw["instruction"])
        task.reward = int(raw["reward"])
        task.work_dir = work if work.is_dir() else task_dir
        feats = features.extract(task)
        meta = manifest.get(task.task_id, {})
        hard = features.structural_skip_reason(task, feats)
        rows.append({
            **feats,
            "task_id": task.task_id,
            "task_type": task.task_type,
            "reward": task.reward,
            "solvable": bool(meta.get("solvable", False)),
            "hard_skip": hard is not None,
            "hard_reason": hard or "",
            "cost_hat": features.heuristic_cost(task, feats),
        })
    return rows


train_rows = load_pack(PACKS["train"]) + load_pack(PACKS["abstention"])
test_rows = load_pack(PACKS["test"])
print(f"train-mix {len(train_rows)} solvable {sum(r['solvable'] for r in train_rows)}")
print(f"test-100 {len(test_rows)} solvable {sum(r['solvable'] for r in test_rows)}")


<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\W'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\*'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\.'
<unknown>:2: SyntaxWarning: invalid escape sequence '\.'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'


train-mix 3100 solvable 2400
test-100 100 solvable 80


In [9]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

NAMES = list(features.FEATURE_NAMES)
train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)


def _journal_paths():
    found = []
    repo = _repo_root()
    candidates = [
        Path("/kaggle/input/cost-journal/cost_journal.jsonl"),
        Path("cost_journal.jsonl"),
        Path("out/cost_journal.jsonl"),
    ]
    if repo is not None:
        candidates.extend([
            repo / "out" / "cost_journal.jsonl",
            repo / "trajectories" / "cost_journal.jsonl",
        ])
    for path in candidates:
        if path.is_file():
            found.append(path)
    return found


costs = {}
for path in _journal_paths():
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            task_id = str(row.get("task_id") or "")
            if task_id and row.get("cost") is not None:
                costs[task_id] = int(row["cost"])
    print("journal", path, "rows", len(costs))

for frame in (train_df, test_df):
    frame["cost_actual"] = frame["task_id"].map(costs)

labeled = train_df[train_df["cost_actual"].notna() & (train_df["cost_actual"] > 0)]
if labeled.empty:
    print("No cost journal. cost.joblib was NOT written. Simulation uses the 800/2500 heuristic.")
    cost_model = None
else:
    cost_model = HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.08,
        max_iter=200,
        random_state=0,
    )
    cost_model.fit(labeled[NAMES], labeled["cost_actual"])
    OUT = out_dir()
    COST_PATH = OUT / "cost.joblib"
    joblib.dump(
        {
            "model": cost_model,
            "feature_names": NAMES,
            "schema_version": features.FEATURE_SCHEMA_VERSION,
        },
        COST_PATH,
    )
    print("wrote", COST_PATH, "rows", len(labeled))
    for frame in (train_df, test_df):
        frame["cost_hat"] = np.clip(cost_model.predict(frame[NAMES]), 1, None).round().astype(int)


No cost journal. cost.joblib was NOT written. Simulation uses the 800/2500 heuristic.


In [10]:
tau = 0.0
p_model = None


def _find_artifact(name: str) -> Path | None:
    candidates = [out_dir() / name, Path.cwd() / name, Path(name)]
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        candidates.extend(sorted(kaggle.rglob(name)))
    for path in candidates:
        if path.is_file():
            return path
    return None


sol_path = _find_artifact("solvability.joblib")
tau_path = _find_artifact("tau.json")
if sol_path is not None:
    blob = joblib.load(sol_path)
    if blob.get("schema_version") == features.FEATURE_SCHEMA_VERSION and list(blob.get("feature_names")) == NAMES:
        p_model = blob["model"]
        print("loaded", sol_path)
    else:
        print("ignored incompatible", sol_path)
if tau_path is not None:
    tau = float(json.loads(tau_path.read_text(encoding="utf-8")).get("tau", 0.0))
    print("tau", tau, "from", tau_path)
if p_model is None:
    print("No solvability model. p=1 except structural skips.")
    print("Attach solvability.joblib from train_solvability Output, or run that notebook first in the same session.")
    for frame in (train_df, test_df):
        frame["p"] = 1.0
else:
    idx = list(p_model.classes_).index(1)
    for frame in (train_df, test_df):
        frame["p"] = p_model.predict_proba(frame[NAMES])[:, idx]


loaded /kaggle/input/models/danbrag/hgb-huawei-classificator/scikitlearn/default/1/solvability.joblib
tau 0.49949168660167664 from /kaggle/input/models/danbrag/hgb-huawei-classificator/scikitlearn/default/1/tau.json


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  ):
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator _BinMapper from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  ):
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator HistGradientBoostingClassifier from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid result

## Metrics


In [11]:
import random


def policy_skip(row, tau):
    if row.task_type == "math":
        return False
    if bool(row.hard_skip):
        return True
    return float(row.p) < tau


def simulate(rows, tau, budget=100_000):
    pending = []
    skipped = 0
    for row in rows.itertuples(index=False):
        if policy_skip(row, tau):
            skipped += 1
            continue
        pending.append(row)
    pending.sort(key=lambda row: -(float(row.p) * float(row.reward) / max(float(row.cost_hat), 1.0)))
    spent = 0
    reward = 0
    wasted = 0
    for row in pending:
        cost = int(row.cost_hat)
        if spent + cost > budget:
            continue
        spent += cost
        if bool(row.solvable):
            reward += int(row.reward)
        else:
            wasted += 1
    return {"reward": reward, "spent": spent, "skipped": skipped, "wasted_attempts": wasted}


def report_cost(title, df):
    print(f"\n== {title} ==")
    known = df[df["cost_actual"].notna()]
    if known.empty or cost_model is None:
        print("MAE: no journal rows for this split")
    else:
        err = (known["cost_hat"] - known["cost_actual"]).abs()
        print(f"MAE all={err.mean():.1f}")
        for task_type, part in known.groupby("task_type"):
            part_err = (part["cost_hat"] - part["cost_actual"]).abs()
            print(f"  {task_type}: MAE={part_err.mean():.1f} n={len(part)}")
    if len(df) <= 120:
        stats = simulate(df, tau)
        print(
            f"budget 100000 once: reward={stats['reward']} spent={stats['spent']} "
            f"skipped={stats['skipped']} wasted_attempts={stats['wasted_attempts']}"
        )
        return
    rng = random.Random(0)
    draws = []
    index = list(df.index)
    for _ in range(30):
        picked = rng.sample(index, 100)
        draws.append(simulate(df.loc[picked], tau))
    rewards = [item["reward"] for item in draws]
    spent = [item["spent"] for item in draws]
    print(
        f"budget 100000 x30 draws of 100: reward mean={sum(rewards)/len(rewards):.1f} "
        f"min={min(rewards)} spent mean={sum(spent)/len(spent):.1f}"
    )


report_cost("train-mix", train_df)
report_cost("test-100", test_df)



== train-mix ==
MAE: no journal rows for this split
budget 100000 x30 draws of 100: reward mean=232.7 min=206 spent mean=97534.6

== test-100 ==
MAE: no journal rows for this split
budget 100000 once: reward=223 spent=99298 skipped=14 wasted_attempts=2


## Download


In [12]:
from IPython.display import display, FileLink, Markdown

OUT = out_dir()
paths = [OUT / "cost.joblib"]
lines = ["## Download artifacts", "", f"Directory: `{OUT}`", ""]
for path in paths:
    if path.is_file():
        lines.append(f"- `{path.name}` ({path.stat().st_size} bytes)")
        display(FileLink(str(path), result_html_prefix=f"{path.name}: "))
    else:
        lines.append(f"- `{path.name}` was not written (no cost journal). Heuristic cost remains in the harness.")
lines.append("")
lines.append("Also download `solvability.joblib` and `tau.json` from the solvability notebook Output.")
lines.append("Put weights into `models/` (or later `artifacts/models` + `RANK_MODELS`).")
display(Markdown("\n".join(lines)))


## Download artifacts

Directory: `/kaggle/working`

- `cost.joblib` was not written (no cost journal). Heuristic cost remains in the harness.

Also download `solvability.joblib` and `tau.json` from the solvability notebook Output.
Put weights into `models/` (or later `artifacts/models` + `RANK_MODELS`).